## A -- Setup & Install

In [1]:
import subprocess, sys, importlib, importlib.util, types

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'], check=False)

pip('typing-extensions==4.15.0')
spec = importlib.util.find_spec('typing_extensions')
if spec:
    _new = types.ModuleType('typing_extensions')
    exec(open(spec.origin).read(), _new.__dict__)
    sys.modules['typing_extensions'] = _new

pip('pydantic-core==2.27.2', 'pydantic==2.10.6')
pip('timm', 'roboflow')
pip('torchmetrics')
pip('albumentations==1.4.24')

for _prefix in ('albumentations', 'pydantic'):
    for k in list(sys.modules.keys()):
        if k.startswith(_prefix):
            del sys.modules[k]

import os, random, time, math, json, csv, warnings, traceback, zipfile, shutil, hashlib, yaml
import numpy as np
import cv2
cv2.setNumThreads(0)   # FIX: cegah deadlock cv2 internal-thread vs DataLoader num_workers
import torch
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path
from collections import defaultdict, Counter
from torch.utils.data import Dataset, DataLoader
import timm
from torchmetrics.detection import MeanAveragePrecision
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f'torch : {torch.__version__}')
print(f'timm  : {timm.__version__}')
print(f'device: {DEVICE}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f}GB)')
print('Setup done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 28.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.29.3 requires pydantic>=2.11.4, but you have pydantic 2.10.6 which is incompatible.
sigstore-models 0.0.6 requires pydantic>=2.12, but you have pydantic 2.10.6 which is incompatible.
a2a-sdk 0.3.26 requires pydantic>=2.11.3, but you have pydantic 2.10.6 which is incompatible.
mcp 1.27.0 requires pydantic<3.0.0,>=2.11.0, but you have pydantic 2.10.6 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.10.6 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 6.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


torch : 2.10.0+cu128
timm  : 1.0.26
device: cuda
  GPU 0: Tesla T4 (14.6GB)
  GPU 1: Tesla T4 (14.6GB)
Setup done.


## B -- Config (v4)

In [2]:
WORK_DIR = Path('/kaggle/working')
RAW_DIR  = WORK_DIR / 'raw_sources'
DATA_DIR = WORK_DIR / 'dataset_localizer_v4'
OUT      = WORK_DIR / 'outputs'
for d in [RAW_DIR, DATA_DIR, OUT]: d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224

# -- Model identity -----------------------------------------------------------
VERSION       = 'v4'
MODEL_NAME    = f'Localizer_MobileNetV4_SingleClass_{VERSION}'
# FASE 3 (ablasi): ganti manual ke 'mobilenetv4_conv_small.e2400_r224_in1k' untuk
# Run C. Jangan otomatis di-loop di notebook ini -- tiga run dijalankan terpisah,
# tiap run kernel-restart bersih, biar tidak ada state training bocor antar-run.
BACKBONE_NAME = 'mobilenetv4_conv_medium'
USE_CBAM      = True   # Run B (ablasi Fase 3) set False, backbone tetap medium

# Single class: localizer cuma cari "ada uang", bukan nominal.
CLASS_NAMES  = ['uang']
NUM_CLASSES  = 1

# -- Training -------------------------------------------------------------------
BATCH_SIZE      = 32
NUM_WORKERS     = 8
EPOCHS_STAGE1   = 20
EPOCHS_STAGE2   = 40
PATIENCE        = 10
MIN_EPOCHS_S1   = 8
MIN_EPOCHS_S2   = 12
LR              = 3e-4
LR_S2           = 5e-5
WEIGHT_DECAY    = 1e-4

LAMBDA_COORD    = 5.0
LAMBDA_NOOBJ    = 0.5
POS_IOU_IGNORE  = 0.3

CONF_THRESHOLD  = 0.5
NMS_IOU_THRESH  = 0.4

# -- Session-split params -----------------------------------------------------
VAL_FRAC  = 0.10
TEST_FRAC = 0.10

# FASE 1, Bagian 6: default False. Roboflow split bawaan itu random per-berkas,
# dataset ini berasal dari video yang dipecah jadi frame -- random per-berkas =
# frame N di train, frame N+1 di test. Persis kebocoran yang sudah diperbaiki
# lewat session-split di v13 (classifier) dan v3 (localizer). Kalau mau buktikan
# sendiri: jalankan sekali dengan True, baca output assert kebocoran session di
# Cell E. Nol -> aman dipakai. Ratusan -> jangan.
RESPECT_SOURCE_SPLIT = False

# FASE 1, item 6 di plan: MIN_BOX_AREA_DEFAULT lama (0.008, ~20x20px di 224)
# berpotensi membuang koin kecil yang justru paling penting buat localizer.
# Sumber koin dapat floor lebih rendah.
MIN_BOX_AREA_DEFAULT = 0.008
MIN_BOX_AREA_COIN    = 0.004

MIN_BOX_AREA_OVERRIDE = {
    # 'kertas_final_rb': diisi setelah inspeksi visual (lihat Cell C3) --
    # kalau box konfirmasi cuma nempel di angka bukan whole-note, source ini
    # DIBUANG total dari localizer (lihat KEEP_SOURCES di bawah), bukan
    # ditambal pakai MIN_BOX_AREA seperti classifier.
}

# -- Sumber data ---------------------------------------------------------------
# Dibuang total dari localizer v4 (confirmed rusak di classifier v14, localizer
# v3 masih memuatnya secara keliru -- lihat plan bagian 3.4):
#   kertas_valid, kertas_money_q0jko
#
# kertas_final_rb: status PENDING sampai Cell C3 (inspeksi visual) selesai.
# Default disertakan; ubah KEEP_SOURCES di bawah kalau hasil inspeksi
# mengonfirmasi box-nya cuma nempel di angka.
KEEP_SOURCES = {
    'koin_1000_tambahan', 'koin_1000_v3', 'koin_fix2', 'koin_detector_ai',
    'kertas_2016_2022', 'kertas_indonesia', 'kertas_final_rb',
}

# -- Paths ------------------------------------------------------------------------
PTH_STAGE1  = OUT / f'localizer_stage1_{VERSION}.pth'
PTH_STAGE2  = OUT / f'localizer_stage2_{VERSION}.pth'
ONNX_PATH   = OUT / f'localizer_{VERSION}.onnx'
TFLITE_PATH = OUT / f'localizer_{VERSION}.tflite'
TFLITE_INT8_PATH = OUT / f'localizer_{VERSION}_int8.tflite'
CSV_PATH    = OUT / f'results_localizer_{VERSION}.csv'
MAP_HIST_PATH = OUT / f'map_diag_{VERSION}.png'

print(f'MODEL_NAME    : {MODEL_NAME}')
print(f'BACKBONE_NAME : {BACKBONE_NAME}')
print(f'USE_CBAM      : {USE_CBAM}')
print(f'IMG_SIZE      : {IMG_SIZE}')
print(f'RESPECT_SOURCE_SPLIT : {RESPECT_SOURCE_SPLIT}')

# -- CHANGELOG v3 -> v4 (lihat localizer-v4-plan.md untuk detail lengkap) -----
# FASE 0: evaluate() lama diganti mAP asli (torchmetrics), ditambah eval
#   end-to-end (localizer -> crop persis kode app -> classifier v15).
#   evaluate() lama dipertahankan sebagai evaluate_teacher_forced() untuk
#   perbandingan historis dengan angka v2/v3, TIDAK dipakai sebagai metrik utama.
# FASE 1: drop kertas_valid + kertas_money_q0jko. Tambah 4 sumber baru
#   (KEEP_CLASSES_NEW_SOURCES wajib diisi manual dari hasil inspeksi Cell C2,
#   TIDAK ditebak dari nama kelas). Port split_by_session_final() greedy
#   load-balancing dari classifier v15 (v3 masih pakai versi lama, session
#   terbesar hardcode ke train).
# FASE 2: parameterisasi box diganti sigmoid + cell-offset (v3 pakai koordinat
#   absolut linear -- salah secara arsitektural untuk conv yang
#   translation-equivariant, lihat plan bagian 3.1). Box loss diganti CIoU
#   (v3 pakai MSE+sqrt ala YOLOv1, tidak selaras dengan metrik IoU).


MODEL_NAME    : Localizer_MobileNetV4_SingleClass_v4
BACKBONE_NAME : mobilenetv4_conv_medium
USE_CBAM      : True
IMG_SIZE      : 224
RESPECT_SOURCE_SPLIT : False


## C -- Dataset Download (sumber lama bersih + 4 sumber baru)

In [3]:
try:
    from kaggle_secrets import UserSecretsClient
    sec    = UserSecretsClient()
    RF_KEY = sec.get_secret('ROBOFLOW_API_KEY')
    print('RF key: Kaggle Secrets')
except Exception:
    RF_KEY = ''  # isi sendiri: app.roboflow.com -> Settings -> API Keys -> Private API Key
    print('RF key: hardcode fallback')

from roboflow import Roboflow
rf = Roboflow(api_key=RF_KEY)

# Sumber lama yang dipertahankan (kertas_valid & kertas_money_q0jko DIBUANG --
# lihat changelog Cell B).
KOIN_SOURCES = [
    ('koin_1000_tambahan', 'try3-mqzxg',      'seribu-rupiah-koin',     1),
    ('koin_1000_v3',       'try7-beyol',       'seribu-rupiah-koin3',    1),
    ('koin_fix2',          'fix2',             'uang-2016',              1),
    ('koin_detector_ai',   'ai-project-1kgr2', 'coin-detector-hkhvv',    9),
]
KERTAS_SOURCES = [
    ('kertas_2016_2022',   'fix-tmc9q',        'uang-2016-2022',         3),
    ('kertas_final_rb',    'final-12',          'final-rb',              3),
    ('kertas_indonesia',   'indonesia-rupiah-currency-dataset', 'indonesia-rupiah-detection', 3),
]

# Sumber BARU (workspace/project/version diparsing langsung dari URL Roboflow
# yang dikasih user, BUKAN ditebak):
#   https://universe.roboflow.com/coin-new/uang-rupiah-kertas-dan-koin-by-adelia-clarissa-zzt2j/dataset/11
#   https://universe.roboflow.com/arfan-faris-pramatya/koin-rupiah-mvpr/dataset/1
#   https://universe.roboflow.com/project-6ajl7/deteksi-koin-djjhw/dataset/2
#   https://universe.roboflow.com/arbiyan-workspace/mln-w2-f6db3/dataset/5
NEW_SOURCES = [
    ('new_adelia_koin_kertas', 'coin-new',             'uang-rupiah-kertas-dan-koin-by-adelia-clarissa-zzt2j', 11),
    ('new_koin_mvpr',          'arfan-faris-pramatya',  'koin-rupiah-mvpr',                                     1),
    ('new_deteksi_koin',       'project-6ajl7',         'deteksi-koin-djjhw',                                   2),
    ('new_mln_w2',             'arbiyan-workspace',     'mln-w2-f6db3',                                         5),
]

ALL_SOURCES = KOIN_SOURCES + KERTAS_SOURCES + NEW_SOURCES

for name, ws, proj, ver in ALL_SOURCES:
    dst = str(RAW_DIR / name)
    if not os.path.isdir(dst):
        try:
            rf.workspace(ws).project(proj).version(ver).download('yolov8', location=dst)
            print(f'  Downloaded : {name}')
        except Exception as e:
            print(f'  SKIP {name}: {e}')
    else:
        print(f'  Exists     : {name}  (WARNING: cache lama -- hapus folder kalau mau force re-download)')


RF key: hardcode fallback
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw_sources/koin_1000_tambahan in yolov8:: 100%|██████████| 1106/1106 [00:00<00:00, 12072.70it/s]

  Downloaded : koin_1000_tambahan
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw_sources/koin_1000_v3 in yolov8:: 100%|██████████| 1112/1112 [00:00<00:00, 11905.57it/s]

...


Extracting Dataset Version Zip to /kaggle/working/raw_sources/new_deteksi_koin in yolov8:: 100%|██████████| 171/171 [00:00<00:00, 5315.15it/s]

  Downloaded : new_deteksi_koin
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/raw_sources/new_mln_w2 in yolov8:: 100%|██████████| 892/892 [00:00<00:00, 12041.04it/s]

  Downloaded : new_mln_w2


## C2 -- WAJIB: inspeksi kelas sumber baru sebelum Cell D (FASE 1)

In [4]:
# WAJIB dijalankan dan dibaca SEBELUM Cell D. Ini yang jadi blocker di rencana --
# nama kelas Roboflow tidak reliable (lihat kertas_2016_2022 index4 "500rp" yang
# ternyata Rp5000_kertas), jadi KEEP_CLASSES_NEW_SOURCES di Cell D TIDAK BOLEH
# ditebak dari sini doang -- cek juga sample gambarnya di bawah.

import yaml
from IPython.display import display as ipy_display, Image as IPyImage

NEW_SOURCE_NAMES = [n for n, *_ in NEW_SOURCES]

for src_name in NEW_SOURCE_NAMES:
    src_dir = RAW_DIR / src_name
    yaml_path = src_dir / 'data.yaml'
    if not yaml_path.exists():
        print(f'{src_name}: data.yaml TIDAK KETEMU, cek download.')
        continue
    names = yaml.safe_load(open(yaml_path)).get('names', [])
    print(f'\n{src_name}: {len(names)} kelas')
    for i, n in enumerate(names):
        print(f'  {i}: {n}')

print('\n' + '='*60)
print('Sekarang cek beberapa sample gambar per kelas per source (biar tidak cuma')
print('nebak dari nama). Ganti src_name / n_per_class di bawah sesuai kebutuhan.')
print('='*60)

def show_samples(src_name, n_per_class=3):
    src_dir = RAW_DIR / src_name
    img_dirs = list(src_dir.rglob('images'))
    yaml_path = src_dir / 'data.yaml'
    names = yaml.safe_load(open(yaml_path)).get('names', []) if yaml_path.exists() else []
    per_class_shown = Counter()
    for idir in img_dirs:
        ldir = idir.parent / 'labels'
        for img_path in sorted(idir.glob('*'))[:2000]:
            lbl_path = ldir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            lines = open(lbl_path).read().splitlines()
            if not lines:
                continue
            cls_id = int(lines[0].split()[0])
            cls_name = names[cls_id] if cls_id < len(names) else str(cls_id)
            if per_class_shown[cls_name] >= n_per_class:
                continue
            per_class_shown[cls_name] += 1
            print(f'{src_name} / kelas {cls_id} ({cls_name}) / {img_path.name}')
            ipy_display(IPyImage(filename=str(img_path), width=200))

# Contoh pemakaian -- uncomment & jalankan manual per source:
# show_samples('new_adelia_koin_kertas')
# show_samples('new_koin_mvpr')
# show_samples('new_deteksi_koin')
# show_samples('new_mln_w2')



new_adelia_koin_kertas: 9 kelas
  0: 1000
  1: 100rb
  2: 10rb
  3: 1rb
  4: 20rb
  5: 2rb
  6: 500
  7: 50rb
  8: 5rb

new_koin_mvpr: 3 kelas
  0: 100
  1: 200
  2: 500

new_deteksi_koin: 8 kelas
  0: 100
  1: 100 Rupiah
  2: 1000
  3: 1000 Rupiah
  4: 200 Rupiah
  5: 50 Rupiah
  6: 500 Rupiah
  7: Balik Koin

new_mln_w2: 3 kelas
  0: koin 100
  1: koin 200
  2: koin 500

Sekarang cek beberapa sample gambar per kelas per source (biar tidak cuma
nebak dari nama). Ganti src_name / n_per_class di bawah sesuai kebutuhan.


## C3 -- Inspeksi visual kertas_final_rb (plan bagian 3.5)

In [5]:
# Plan bagian 3.5: kertas_final_rb override MIN_BOX_AREA=0.10 di classifier
# dengan alasan "box nempel teks angka doang, bukan whole-note". Buat classifier
# itu masih tertolong (crop angka tetap bisa diklasifikasi), buat localizer itu
# racun -- localizer belajar naruh box di angka, bukan di seluruh lembar uang.
#
# Jalankan, lihat 12 sample di bawah. Kalau box (garis hijau) memeluk SELURUH
# lembar uang -> aman, biarkan di KEEP_SOURCES (Cell B). Kalau box cuma
# nempel di kotak angka nominal -> hapus 'kertas_final_rb' dari KEEP_SOURCES.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

src_dir = RAW_DIR / 'kertas_final_rb'
img_dirs = list(src_dir.rglob('images'))
samples = []
for idir in img_dirs:
    ldir = idir.parent / 'labels'
    for img_path in sorted(idir.glob('*')):
        lbl_path = ldir / (img_path.stem + '.txt')
        if lbl_path.exists() and open(lbl_path).read().strip():
            samples.append((img_path, lbl_path))
        if len(samples) >= 12:
            break
    if len(samples) >= 12:
        break

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, (img_path, lbl_path) in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    ax.imshow(img)
    for line in open(lbl_path).read().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        cx, cy, bw, bh = map(float, p[1:5])
        x, y = (cx - bw/2) * W, (cy - bh/2) * H
        rect = mpatches.Rectangle((x, y), bw*W, bh*H, linewidth=2,
                                    edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
    ax.axis('off')
plt.tight_layout()
plt.savefig(str(OUT / 'kertas_final_rb_inspect.png'), dpi=100)
plt.show()
print('Kalau box cuma nempel angka: KEEP_SOURCES.discard(\'kertas_final_rb\') di Cell B, lalu re-run dari Cell B.')


Kalau box cuma nempel angka: KEEP_SOURCES.discard('kertas_final_rb') di Cell B, lalu re-run dari Cell B.


## D -- KEEP_CLASSES gate untuk sumber baru (isi manual, jangan ditebak)

In [7]:
# Aturan mapping (dari chat, dikunci): kelas yang nama/anotasinya mengandung
# nominal (100, 200, 500, ...) dipakai; varian sisi ("200 depan"/"200 belakang")
# tetap dipakai, semuanya collapse ke kelas 0 ("uang") karena localizer cuma
# cari ada/tidaknya uang. Kelas TANPA nominal (background, objek lain, koin
# non-Rupiah) DIBUANG -- gambarnya tidak diikutsertakan sama sekali, bukan
# cuma label-nya yang di-drop.

KEEP_CLASSES_NEW_SOURCES = {
    'new_adelia_koin_kertas': {0, 1, 2, 3, 4, 5, 6, 7, 8},
    # 9 kelas: 1000, 100rb, 10rb, 1rb, 20rb, 2rb, 500, 50rb, 5rb -- semua ada
    # nominal, tidak ada kelas non-uang, semua dipakai.

    'new_koin_mvpr': {0, 1, 2},
    # 100, 200, 500 -- semua ada nominal.

    'new_deteksi_koin': {0, 1, 2, 3, 4, 5, 6},
    # index 7 "Balik Koin" DIBUANG -- tidak ada nominal (beda dari kasus
    # "200 depan"/"200 belakang" yang nominalnya jelas). index 0 & 1
    # sama-sama "100" (satu polos, satu "100 Rupiah") -- tetap dipakai
    # dua-duanya, cuma penamaan sumbernya tidak konsisten.

    'new_mln_w2': {0, 1, 2},
    # koin 100, koin 200, koin 500 -- semua ada nominal.
}

_unfilled = [k for k, v in KEEP_CLASSES_NEW_SOURCES.items() if v is None]
if _unfilled:
    raise ValueError(
        f'KEEP_CLASSES_NEW_SOURCES belum diisi untuk: {_unfilled}. '
        f'Baca output Cell C2 dulu (nama kelas + sample gambar), baru isi dict '
        f'di atas dengan index kelas yang valid duit Rupiah. Jangan ditebak.'
    )

print('KEEP_CLASSES_NEW_SOURCES OK:')
for k, v in KEEP_CLASSES_NEW_SOURCES.items():
    print(f'  {k}: keep index {sorted(v)}')

KEEP_CLASSES_NEW_SOURCES OK:
  new_adelia_koin_kertas: keep index [0, 1, 2, 3, 4, 5, 6, 7, 8]
  new_koin_mvpr: keep index [0, 1, 2]
  new_deteksi_koin: keep index [0, 1, 2, 3, 4, 5, 6]
  new_mln_w2: keep index [0, 1, 2]


## E -- Dedup + Collapse ke Single Class + Session Split (greedy load-balancing, FASE 1)

In [8]:
import re

# Urutan prioritas dedup: source duluan di list = dipertahankan kalau ketemu
# gambar/session duplikat lintas source.
SOURCE_PRIORITY = [
    'kertas_final_rb', 'kertas_2016_2022', 'kertas_indonesia',
    'koin_detector_ai', 'koin_1000_v3', 'koin_1000_tambahan', 'koin_fix2',
    'new_adelia_koin_kertas', 'new_koin_mvpr', 'new_deteksi_koin', 'new_mln_w2',
]
SOURCE_PRIORITY = [s for s in SOURCE_PRIORITY if s in KEEP_SOURCES or s.startswith('new_')]

def get_min_area(src_name):
    if src_name in MIN_BOX_AREA_OVERRIDE:
        return MIN_BOX_AREA_OVERRIDE[src_name]
    if src_name.startswith('koin') or src_name.startswith('new_'):
        return MIN_BOX_AREA_COIN
    return MIN_BOX_AREA_DEFAULT

def file_hash(p):
    return hashlib.md5(open(p, 'rb').read()).hexdigest()

def session_id_v2(fname):
    stem = re.sub(r'_jpg\.rf\..*$', '', fname)
    m = re.match(r'^(.*mp4)-?\d*$', stem, re.IGNORECASE)
    if m:
        return m.group(1)
    return stem

print('-- Scanning semua source, dedup by image hash --')
seen_hash = {}
n_total, n_dup, n_class_filtered = 0, 0, 0

for src_name in SOURCE_PRIORITY:
    src_dir = RAW_DIR / src_name
    if not src_dir.exists():
        print(f'  SKIP {src_name}: folder tidak ada')
        continue
    keep_classes = KEEP_CLASSES_NEW_SOURCES.get(src_name)  # None utk source lama = keep semua
    img_dirs = list(src_dir.rglob('images'))
    for idir in img_dirs:
        ldir = idir.parent / 'labels'
        for img_path in idir.glob('*'):
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            label_path = ldir / (img_path.stem + '.txt')
            if not label_path.exists():
                continue
            n_total += 1
            h = file_hash(img_path)
            if h in seen_hash:
                n_dup += 1
                continue
            seen_hash[h] = (src_name, img_path, label_path, keep_classes)

print(f'Total file ditemukan : {n_total}')
print(f'Duplicate dibuang    : {n_dup}')
print(f'Unique dipakai       : {len(seen_hash)}')

ALL_IMG_DIR = DATA_DIR / 'all' / 'images'
ALL_LBL_DIR = DATA_DIR / 'all' / 'labels'
ALL_IMG_DIR.mkdir(parents=True, exist_ok=True)
ALL_LBL_DIR.mkdir(parents=True, exist_ok=True)

unified_records = []   # (dst_img, dst_lbl, src_name, orig_name, orig_split)
n_img_zero_box = 0
zero_box_per_src = Counter()
box_filtered_per_src = Counter()

for h, (src_name, img_path, label_path, keep_classes) in seen_hash.items():
    new_stem = h[:16]
    dst_img = ALL_IMG_DIR / f'{new_stem}{img_path.suffix.lower()}'
    dst_lbl = ALL_LBL_DIR / f'{new_stem}.txt'
    if not dst_img.exists():
        shutil.copy2(str(img_path), str(dst_img))

    min_area = get_min_area(src_name)
    lines_out = []
    n_lines_raw = 0
    for line in open(label_path).read().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            continue
        n_lines_raw += 1
        cls_id, cx, cy, w, bh = parts
        if keep_classes is not None and int(cls_id) not in keep_classes:
            n_class_filtered += 1
            continue
        w_f, bh_f = float(w), float(bh)
        if w_f * bh_f < min_area:
            box_filtered_per_src[src_name] += 1
            continue
        lines_out.append(f'0 {cx} {cy} {w} {bh}')   # collapse ke class 0 = uang

    if n_lines_raw == 0:
        n_img_zero_box += 1
        zero_box_per_src[src_name] += 1

    if lines_out:
        dst_lbl.write_text('\n'.join(lines_out))
        # orig_split dipakai kalau RESPECT_SOURCE_SPLIT=True; parent.parent.name
        # = 'train'/'valid'/'test' sesuai struktur folder download Roboflow.
        orig_split = img_path.parent.parent.name
        unified_records.append((dst_img, dst_lbl, src_name, img_path.name, orig_split))

print(f'\nKelas difilter (bukan Rupiah / tanpa nominal): {n_class_filtered}')
print(f'Box difilter (area < threshold per-source):')
for src, cnt in box_filtered_per_src.most_common():
    print(f'  {src:<28}: {cnt}')
print(f'\nGambar zero-box: {n_img_zero_box}')
print(f'\nUnified dataset: {len(unified_records)} gambar (single class "uang")')

if RESPECT_SOURCE_SPLIT:
    # Bukan default. Lihat alasan di Cell B. Kalau dipakai, assert kebocoran
    # session di bawah AKAN melapor -- baca angkanya sebelum lanjut training.
    print('\nRESPECT_SOURCE_SPLIT=True -- pakai folder train/valid/test asli Roboflow.')
    train, valid, test = [], [], []
    for rec in unified_records:
        split_map = {'train': train, 'valid': valid, 'test': test, 'val': valid}
        split_map.get(rec[4], train).append(rec)
else:
    # -- FASE 1: port split_by_session_final() greedy load-balancing dari
    # classifier v15. v3 lama hardcode session terbesar ke train tanpa syarat
    # lalu alokasi sisa session all-or-nothing berurutan -- overshoot kalau ada
    # 1 session besar (lihat plan bagian 3.6). Versi ini alokasikan tiap session
    # ke split yang defisitnya paling besar saat itu.
    sessions = defaultdict(list)
    for rec in unified_records:
        dst_img, dst_lbl, src_name, orig_name, orig_split = rec
        sessions[session_id_v2(orig_name)].append(rec)

    session_list = sorted(sessions.items(), key=lambda kv: len(kv[1]), reverse=True)
    n_sessions = len(session_list)
    n = len(unified_records)
    print(f'\nTotal unique session: {n_sessions} dari {n} gambar')

    if n_sessions < 3:
        print('  WARNING: sesi terlalu sedikit -- fallback frame-level random split.')
        shuffled = unified_records.copy()
        random.shuffle(shuffled)
        n_val = max(1, round(n * VAL_FRAC))
        n_tst = max(1, round(n * TEST_FRAC))
        train, valid, test = shuffled[n_val+n_tst:], shuffled[:n_val], shuffled[n_val:n_val+n_tst]
    else:
        train_frac = 1.0 - VAL_FRAC - TEST_FRAC
        targets = {'train': n * train_frac, 'valid': n * VAL_FRAC, 'test': n * TEST_FRAC}
        counts  = {'train': 0, 'valid': 0, 'test': 0}
        buckets = {'train': [], 'valid': [], 'test': []}

        biggest_session_size = len(session_list[0][1])
        for _, v in session_list:
            deficits = {k: targets[k] - counts[k] for k in targets}
            best = max(deficits, key=deficits.get)
            buckets[best].extend(v)
            counts[best] += len(v)

        dominance_ratio = biggest_session_size / n
        if dominance_ratio > 0.25:
            print(f'  WARNING: session terbesar = {biggest_session_size}/{n} '
                  f'({dominance_ratio:.1%}) -- dominasi tinggi, cek manual.')

        train, valid, test = buckets['train'], buckets['valid'], buckets['test']

splits = {'train': train, 'val': valid, 'test': test}
for split_name, records in splits.items():
    img_dir = DATA_DIR / split_name / 'images'
    lbl_dir = DATA_DIR / split_name / 'labels'
    if img_dir.exists(): shutil.rmtree(str(img_dir))
    if lbl_dir.exists(): shutil.rmtree(str(lbl_dir))
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img_p, lbl_p, _src, _orig, _osplit in records:
        shutil.copy2(str(img_p), str(img_dir / img_p.name))
        shutil.copy2(str(lbl_p), str(lbl_dir / lbl_p.name))
    print(f'  {split_name:<6}: {len(records)} gambar')

def hashes_of(split_name):
    return set(file_hash(p) for p in (DATA_DIR/split_name/'images').glob('*'))

h_train, h_val, h_test = hashes_of('train'), hashes_of('val'), hashes_of('test')
overlap_hash = (h_train & h_val) | (h_train & h_test) | (h_val & h_test)
print(f'\nOverlap hash antar split: {len(overlap_hash)} (harus 0)')
assert len(overlap_hash) == 0, 'DATA LEAKAGE -- ada gambar sama di >1 split!'

s_train = {session_id_v2(o) for _, _, _, o, _ in train}
s_val   = {session_id_v2(o) for _, _, _, o, _ in valid}
s_test  = {session_id_v2(o) for _, _, _, o, _ in test}
leak_tv, leak_tt = s_train & s_val, s_train & s_test
print(f'Session leakage train-val : {len(leak_tv)} (harus 0 kalau RESPECT_SOURCE_SPLIT=False)')
print(f'Session leakage train-test: {len(leak_tt)} (harus 0 kalau RESPECT_SOURCE_SPLIT=False)')
if RESPECT_SOURCE_SPLIT and (leak_tv or leak_tt):
    print('  -> INI JAWABAN eksperimennya: session bocor lintas split Roboflow asli. '
        'Set RESPECT_SOURCE_SPLIT=False kalau angka di atas 0.')
elif not RESPECT_SOURCE_SPLIT and (leak_tv or leak_tt):
    print('  WARNING: masih ada leakage padahal session-split aktif -- cek session_id_v2 regex.')
else:
    print('OK: tidak ada session leakage.')


-- Scanning semua source, dedup by image hash --
Total file ditemukan : 32948
Duplicate dibuang    : 138
Unique dipakai       : 32810

Kelas difilter (bukan Rupiah / tanpa nominal): 0
Box difilter (area < threshold per-source):
  kertas_final_rb             : 2443
  koin_fix2                   : 116
  kertas_2016_2022            : 106
  koin_detector_ai            : 88

Gambar zero-box: 9105

Unified dataset: 23581 gambar (single class "uang")

Total unique session: 6844 dari 23581 gambar
  train : 18865 gambar
  val   : 2358 gambar
  test  : 2358 gambar

Overlap hash antar split: 0 (harus 0)
Session leakage train-val : 0 (harus 0 kalau RESPECT_SOURCE_SPLIT=False)
Session leakage train-test: 0 (harus 0 kalau RESPECT_SOURCE_SPLIT=False)
OK: tidak ada session leakage.


## F -- Dataset + DataLoader (bbox-aware augmentation, letterbox resize)

In [9]:
def load_yolo_boxes(label_path):
    '''Return list of [cx, cy, w, h] normalized, class selalu 0.'''
    boxes = []
    for line in open(label_path).read().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            continue
        _, cx, cy, w, h = parts
        boxes.append([float(cx), float(cy), float(w), float(h)])
    return boxes


def build_train_aug(size):
    return A.Compose([
        A.OneOf([
            A.RandomSizedBBoxSafeCrop(width=size, height=size, erosion_rate=0.15, p=1.0),
            A.NoOp(p=1.0),
        ], p=0.4),
        A.LongestMaxSize(max_size=size),
        A.PadIfNeeded(min_height=size, min_width=size,
                      border_mode=0, value=(114, 114, 114)),
        A.HorizontalFlip(p=0.5),
        A.CoarseDropout(num_holes_range=(1, 2),
                         hole_height_range=(int(size*0.1), int(size*0.35)),
                         hole_width_range=(int(size*0.1), int(size*0.35)),
                         fill=114, p=0.25),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=25, val_shift_limit=20, p=0.4),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.08), p=1.0),
            A.ISONoise(p=1.0),
        ], p=0.3),
        A.GaussianBlur(blur_limit=3, p=0.15),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'],
                                  min_visibility=0.15))


def build_eval_aug(size):
    return A.Compose([
        A.LongestMaxSize(max_size=size),
        A.PadIfNeeded(min_height=size, min_width=size,
                      border_mode=0, value=(114, 114, 114)),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'],
                                  min_visibility=0.3))   # eval tetap strict, gak diubah


class LocalizerDataset(Dataset):
    def __init__(self, split, grid_size, transform):
        self.img_dir = DATA_DIR / split / 'images'
        self.lbl_dir = DATA_DIR / split / 'labels'
        self.files = sorted(self.img_dir.glob('*'))
        self.grid_size = grid_size
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        lbl_path = self.lbl_dir / (img_path.stem + '.txt')
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        boxes = load_yolo_boxes(lbl_path)
        class_labels = [0] * len(boxes)

        # clip box biar valid sebelum augment (kadang ada float di luar [0,1] dari sumber)
        clipped = []
        for cx, cy, w, h in boxes:
            cx = min(max(cx, 0.001), 0.999)
            cy = min(max(cy, 0.001), 0.999)
            w  = min(max(w, 0.002), 1.0)
            h  = min(max(h, 0.002), 1.0)
            clipped.append([cx, cy, w, h])

        try:
            out = self.transform(image=image, bboxes=clipped, class_labels=class_labels)
        except Exception:
            out = self.transform(image=image, bboxes=[], class_labels=[])

        img_t = out['image']
        gt_boxes = out['bboxes']

        S = self.grid_size
        target = torch.zeros(S, S, 5)   # [objectness, cx, cy, w, h] per cell
        n_collisions = 0
        for cx, cy, w, h in gt_boxes:
            gi, gj = min(int(cx * S), S - 1), min(int(cy * S), S - 1)
            if target[gj, gi, 0] == 1:
                n_collisions += 1   # cell sudah punya box lain -- limitation grid coarse
                continue
            target[gj, gi, 0] = 1.0
            target[gj, gi, 1] = cx
            target[gj, gi, 2] = cy
            target[gj, gi, 3] = w
            target[gj, gi, 4] = h

        return img_t, target, n_collisions


def collate_fn(batch):
    imgs = torch.stack([b[0] for b in batch])
    targets = torch.stack([b[1] for b in batch])
    collisions = sum(b[2] for b in batch)
    return imgs, targets, collisions

print('Dataset class siap (v3: + occlusion aug, min_visibility train=0.15). '
      'Grid size ditentukan setelah backbone dibangun (cell G).')


Dataset class siap (v3: + occlusion aug, min_visibility train=0.15). Grid size ditentukan setelah backbone dibangun (cell G).


## G -- CBAM (TFLite-safe)

In [10]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=[2, 3], keepdim=True)
        mx  = x.flatten(2).max(dim=2).values.view(
              x.size(0), x.size(1), 1, 1)
        return x * self.sigmoid(self.fc(avg) + self.fc(mx))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        pad = (kernel_size - 1) // 2
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=pad, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True).values
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.sa(self.ca(x))


_x = torch.randn(2, 64, 7, 7)
_y = CBAM(64)(_x)
assert _y.shape == _x.shape
print(f'CBAM OK: {_x.shape} -> {_y.shape}')
del _x, _y

CBAM OK: torch.Size([2, 64, 7, 7]) -> torch.Size([2, 64, 7, 7])


## H -- Model: Backbone + Head, box parameterization diperbaiki (FASE 2 item 1)

In [20]:
class RupiahLocalizer(nn.Module):
    '''
    Backbone (MobileNetV4) -> feature map -> conv head -> grid SxSx5.

    FASE 2 FIX (plan bagian 3.1): v3 mengeluarkan cx,cy,w,h sebagai koordinat
    ABSOLUT linear langsung dari conv. Itu salah secara arsitektural -- conv
    translation-equivariant, jadi pola visual identik di cell manapun
    menghasilkan aktivasi identik, padahal target cx-nya beda jauh tergantung
    posisi cell. Head akhirnya cuma bisa menebak dari artefak padding di tepi.

    Fix: head cuma memprediksi OFFSET relatif dalam cell (translation-invariant,
    cocok sifat conv), lalu di forward() di-decode ke koordinat absolut pakai
    grid_x/grid_y (buffer konstan, bukan parameter -- aman untuk ONNX/TFLite).
    Objectness TETAP raw logit (bukan di-sigmoid di sini) supaya loss bisa
    pakai BCEWithLogits yang stabil secara numerik; sigmoid objectness cuma
    dipasang saat decode/eval dan di export wrapper (persis pola v3).

    Tidak ada NMS di dalam model -- NMS dikerjakan terpisah saat decode/eval,
    TIDAK di-export ke graph.
    '''
    def __init__(self, backbone_name=BACKBONE_NAME, pretrained=True, with_cbam=False):
        super().__init__()
        self.with_cbam = with_cbam

        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained,
            features_only=True, out_indices=(-2,))

        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            feats = self.backbone(dummy)
        feat_channels = feats[-1].shape[1]
        self.grid_size = feats[-1].shape[-1]
        S = self.grid_size

        self.cbam = CBAM(feat_channels) if with_cbam else nn.Identity()

        self.head = nn.Sequential(
            nn.Conv2d(feat_channels, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 5, 1),   # [objectness, tx, ty, tw, th] raw logits per cell
        )

        # grid_y[i,j] = i (baris = index y), grid_x[i,j] = j (kolom = index x).
        # Cocok dengan konvensi target[gj, gi, :] di LocalizerDataset (gj=row/y,
        # gi=col/x). Buffer, bukan parameter -- ikut .to(device)/export, tidak
        # ikut dilatih.
        grid_y, grid_x = torch.meshgrid(
            torch.arange(S, dtype=torch.float32),
            torch.arange(S, dtype=torch.float32), indexing='ij')
        self.register_buffer('grid_x', grid_x.contiguous().clone())   # [S,S]
        self.register_buffer('grid_y', grid_y.contiguous().clone())   # [S,S]
         # [S,S]

        
        print(f'{backbone_name}: feat_channels={feat_channels}  grid={S}x{S}')
        print(f'  with_cbam={with_cbam}')
        total = sum(p.numel() for p in self.parameters())
        print(f'  total_params={total:,}')

    def forward(self, x):
        feats = self.backbone(x)[-1]
        feats = self.cbam(feats)
        raw = self.head(feats)               # [B, 5, S, S]
        raw = raw.permute(0, 2, 3, 1)         # [B, S, S, 5]

        obj_logit = raw[..., 0:1]
        tx, ty, tw, th = raw[..., 1], raw[..., 2], raw[..., 3], raw[..., 4]

        S = self.grid_size
        cx = (torch.sigmoid(tx) + self.grid_x) / S
        cy = (torch.sigmoid(ty) + self.grid_y) / S
        w  = torch.sigmoid(tw)
        h  = torch.sigmoid(th)
        box = torch.stack([cx, cy, w, h], dim=-1)   # [B,S,S,4], sudah absolut 0..1

        return torch.cat([obj_logit, box], dim=-1)  # [B,S,S,5]: (obj_logit_raw, cx,cy,w,h)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
        print('Backbone frozen.')

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
        n = sum(p.numel() for p in self.backbone.parameters())
        print(f'Backbone unfrozen: {n:,} params')


_m = RupiahLocalizer(pretrained=False, with_cbam=True).eval()
with torch.no_grad():
    _out = _m(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE))
GRID_SIZE = _m.grid_size
print(f'Model forward OK: {_out.shape}  GRID_SIZE={GRID_SIZE}')
print(f'  box range check -- cx,cy,w,h harus di [0,1]: '
      f'{_out[...,1:].min().item():.3f} .. {_out[...,1:].max().item():.3f}')
del _m, _out


mobilenetv4_conv_medium: feat_channels=160  grid=14x14
  with_cbam=True
  total_params=7,465,207
Model forward OK: torch.Size([2, 14, 14, 5])  GRID_SIZE=14
  box range check -- cx,cy,w,h harus di [0,1]: 0.037 .. 0.966


## I -- Loss: CIoU box loss (FASE 2 item 2)

In [12]:
def xywh_to_xyxy(box):
    cx, cy, w, h = box[..., 0], box[..., 1], box[..., 2], box[..., 3]
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)


def ciou(pred, target, eps=1e-7):
    '''
    pred, target: [..., 4] = cx,cy,w,h, normalized 0..1. Return CIoU per box.
    Dipakai buat box loss (FASE 2 item 2) -- v3 pakai MSE+sqrt(w,h) ala YOLOv1,
    yang dioptimasi (MSE) beda dari yang dilaporkan (IoU). CIoU optimasi
    langsung terhadap IoU + jarak center + rasio aspek, selaras sama metrik.
    '''
    p_xyxy = xywh_to_xyxy(pred)
    t_xyxy = xywh_to_xyxy(target)

    px1, py1, px2, py2 = p_xyxy.unbind(-1)
    tx1, ty1, tx2, ty2 = t_xyxy.unbind(-1)

    ix1, iy1 = torch.max(px1, tx1), torch.max(py1, ty1)
    ix2, iy2 = torch.min(px2, tx2), torch.min(py2, ty2)
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)

    p_area = (px2 - px1).clamp(min=0) * (py2 - py1).clamp(min=0)
    t_area = (tx2 - tx1).clamp(min=0) * (ty2 - ty1).clamp(min=0)
    union = p_area + t_area - inter + eps
    iou = inter / union

    cx1, cy1 = torch.min(px1, tx1), torch.min(py1, ty1)
    cx2, cy2 = torch.max(px2, tx2), torch.max(py2, ty2)
    c_diag_sq = (cx2 - cx1).pow(2) + (cy2 - cy1).pow(2) + eps

    p_cx, p_cy = pred[..., 0], pred[..., 1]
    t_cx, t_cy = target[..., 0], target[..., 1]
    center_dist_sq = (p_cx - t_cx).pow(2) + (p_cy - t_cy).pow(2)

    p_w, p_h = pred[..., 2].clamp(min=eps), pred[..., 3].clamp(min=eps)
    t_w, t_h = target[..., 2].clamp(min=eps), target[..., 3].clamp(min=eps)
    v = (4 / (math.pi ** 2)) * (torch.atan(t_w / t_h) - torch.atan(p_w / p_h)).pow(2)
    with torch.no_grad():
        alpha = v / (1 - iou + v + eps)

    return iou - center_dist_sq / c_diag_sq - alpha * v


def localizer_loss(pred, target):
    '''
    pred, target: [B, S, S, 5] = (objectness_logit, cx, cy, w, h) sudah dalam
    koordinat absolut 0..1 (decode terjadi di dalam model.forward(), lihat
    Cell H FASE 2 item 1).
    '''
    obj_mask   = target[..., 0] == 1
    noobj_mask = ~obj_mask

    pred_obj   = pred[..., 0]
    target_obj = target[..., 0]

    loss_obj = F.binary_cross_entropy_with_logits(
        pred_obj[obj_mask], target_obj[obj_mask], reduction='sum'
    ) if obj_mask.any() else torch.tensor(0.0, device=pred.device)

    loss_noobj = F.binary_cross_entropy_with_logits(
        pred_obj[noobj_mask], target_obj[noobj_mask], reduction='sum'
    ) if noobj_mask.any() else torch.tensor(0.0, device=pred.device)

    if obj_mask.any():
        pred_box   = pred[..., 1:5][obj_mask]
        target_box = target[..., 1:5][obj_mask]
        ciou_vals = ciou(pred_box, target_box)
        loss_box = (1.0 - ciou_vals).sum()
    else:
        loss_box = torch.tensor(0.0, device=pred.device)

    n_pos = max(obj_mask.sum().item(), 1)
    total = (LAMBDA_COORD * loss_box + loss_obj + LAMBDA_NOOBJ * loss_noobj) / n_pos
    return total, {
        'obj': loss_obj.item() / n_pos,
        'noobj': loss_noobj.item() / n_pos,
        'box_ciou': loss_box.item() / n_pos,
    }

print('Loss function siap (CIoU box loss, FASE 2 item 2).')


Loss function siap (CIoU box loss, FASE 2 item 2).


## J -- Eval utilities: mAP asli (FASE 0) + teacher-forced (pembanding historis)

In [13]:
def iou_xywh(box1, box2):
    b1_x1, b1_y1 = box1[0]-box1[2]/2, box1[1]-box1[3]/2
    b1_x2, b1_y2 = box1[0]+box1[2]/2, box1[1]+box1[3]/2
    b2_x1, b2_y1 = box2[0]-box2[2]/2, box2[1]-box2[3]/2
    b2_x2, b2_y2 = box2[0]+box2[2]/2, box2[1]+box2[3]/2
    ix1, iy1 = max(b1_x1, b2_x1), max(b1_y1, b2_y1)
    ix2, iy2 = min(b1_x2, b2_x2), min(b1_y2, b2_y2)
    iw, ih = max(ix2-ix1, 0), max(iy2-iy1, 0)
    inter = iw * ih
    area1 = max(box1[2]*box1[3], 1e-6)
    area2 = max(box2[2]*box2[3], 1e-6)
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0


def xywh_to_xyxy_np(cx, cy, w, h):
    return [cx - w/2, cy - h/2, cx + w/2, cy + h/2]


def decode_boxes(pred, conf_thresh=CONF_THRESHOLD):
    '''pred: [S,S,5], box sudah absolut (decode terjadi di model.forward()).
    Return list of (conf, cx,cy,w,h) di atas threshold.'''
    obj_prob = torch.sigmoid(pred[..., 0])
    S = obj_prob.shape[0]
    boxes = []
    for gj in range(S):
        for gi in range(S):
            if obj_prob[gj, gi] >= conf_thresh:
                cx, cy, bw, bh = pred[gj, gi, 1:5].tolist()
                boxes.append((obj_prob[gj, gi].item(), cx, cy, bw, bh))
    return boxes


def nms_plain(boxes, iou_thresh=NMS_IOU_THRESH):
    '''NMS standar, IoU-only. FASE 2 item 3: smart_nms (center-distance
    suppression) v3 berisiko menekan 2 koin yang memang bersebelahan di meja --
    skenario nyata di app. Dengan box parameterization sudah benar (Cell H),
    box tidak lagi pecah jadi 2 prediksi untuk 1 objek, jadi NMS biasa cukup.'''
    boxes = sorted(boxes, key=lambda b: b[0], reverse=True)
    keep = []
    while boxes:
        best = boxes.pop(0)
        keep.append(best)
        boxes = [b for b in boxes if iou_xywh(best[1:], b[1:]) < iou_thresh]
    return keep


@torch.no_grad()
def evaluate_teacher_forced(model, loader, conf_thresh=CONF_THRESHOLD):
    '''Metrik lama v2/v3 (plan bagian 3.2). IoU/recall CUMA dihitung di cell
    yang punya GT, false positive TIDAK PERNAH dihitung. Dipertahankan untuk
    perbandingan historis dengan angka v2 (0.7565/0.8943) SAJA -- JANGAN
    dipakai sebagai metrik utama, dan JANGAN dibandingkan langsung dengan
    mAP baseline YOLO (0.9949) di evaluate_map() -- beda protokol sama sekali.'''
    model.eval()
    total_loss, n_batches = 0.0, 0
    ious = []
    n_gt_total, n_detected = 0, 0
    for imgs, targets, _ in loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        preds = model(imgs)
        loss, _ = localizer_loss(preds, targets)
        total_loss += loss.item(); n_batches += 1
        obj_prob = torch.sigmoid(preds[..., 0])
        B, S, _ = obj_prob.shape
        for b in range(B):
            gt_mask = targets[b, ..., 0] == 1
            n_gt_total += gt_mask.sum().item()
            for gi in range(S):
                for gj in range(S):
                    if targets[b, gj, gi, 0] == 1 and obj_prob[b, gj, gi] >= conf_thresh:
                        gt_box   = targets[b, gj, gi, 1:5].cpu().numpy()
                        pred_box = preds[b, gj, gi, 1:5].cpu().numpy()
                        ious.append(iou_xywh(gt_box, pred_box))
                        n_detected += 1
    mean_iou = float(np.mean(ious)) if ious else 0.0
    recall   = n_detected / max(n_gt_total, 1)
    return total_loss / max(n_batches, 1), mean_iou, recall


@torch.no_grad()
def evaluate_map(model, loader, decode_conf=0.01, nms_thresh=NMS_IOU_THRESH):
    '''FASE 0: mAP asli lewat torchmetrics -- pencocokan prediksi<->GT global,
    false positive DIHITUNG. decode_conf rendah (0.01) sengaja dipakai di sini
    (bukan CONF_THRESHOLD=0.5) supaya kurva precision-recall lengkap, sesuai
    cara mAP dihitung standar (beda dari threshold operasional 0.5 yang dipakai
    aplikasi saat inference).'''
    model.eval()
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    preds_list, targets_list = [], []

    for imgs, targets, _ in loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).cpu()
        targets = targets.cpu()
        B, S, _ = preds.shape[0], preds.shape[1], preds.shape[2]

        for b in range(B):
            raw_boxes = decode_boxes(preds[b], conf_thresh=decode_conf)
            kept = nms_plain(raw_boxes, iou_thresh=nms_thresh)
            if kept:
                p_boxes = torch.tensor([xywh_to_xyxy_np(cx, cy, w, h) for _, cx, cy, w, h in kept])
                p_scores = torch.tensor([c for c, *_ in kept])
                p_labels = torch.zeros(len(kept), dtype=torch.long)
            else:
                p_boxes = torch.zeros((0, 4)); p_scores = torch.zeros(0); p_labels = torch.zeros(0, dtype=torch.long)
            preds_list.append({'boxes': p_boxes, 'scores': p_scores, 'labels': p_labels})

            gt_mask = targets[b, ..., 0] == 1
            gt_boxes_xywh = targets[b][gt_mask][:, 1:5]
            if len(gt_boxes_xywh):
                t_boxes = torch.stack([torch.tensor(xywh_to_xyxy_np(*box.tolist())) for box in gt_boxes_xywh])
                t_labels = torch.zeros(len(gt_boxes_xywh), dtype=torch.long)
            else:
                t_boxes = torch.zeros((0, 4)); t_labels = torch.zeros(0, dtype=torch.long)
            targets_list.append({'boxes': t_boxes, 'labels': t_labels})

    metric.update(preds_list, targets_list)
    result = metric.compute()
    return {
        'map50':    result['map_50'].item(),
        'map50_95': result['map'].item(),
        'mar100':   result['mar_100'].item(),
    }


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, n_batches = 0.0, 0
    for imgs, targets, _ in loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss, _ = localizer_loss(preds, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item(); n_batches += 1
    return total_loss / max(n_batches, 1)


print('Eval utilities siap: evaluate_map() (metrik utama), '
      'evaluate_teacher_forced() (pembanding historis SAJA).')


Eval utilities siap: evaluate_map() (metrik utama), evaluate_teacher_forced() (pembanding historis SAJA).


## G2 -- Build DataLoaders

In [14]:
train_ds = LocalizerDataset('train', GRID_SIZE, build_train_aug(IMG_SIZE))
val_ds   = LocalizerDataset('val',   GRID_SIZE, build_eval_aug(IMG_SIZE))
test_ds  = LocalizerDataset('test',  GRID_SIZE, build_eval_aug(IMG_SIZE))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, collate_fn=collate_fn)

print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

# Cek seberapa sering grid collision terjadi (box ke-cell yang sama) di train set.
# FIX v3: sample N batch doang, bukan full epoch -- dataset 39k+ gambar,
# augmentasi berat (RandomSizedBBoxSafeCrop dll), full loop makan waktu lama
# cuma buat itung 1 angka sanity-check.
N_SAMPLE_BATCH = 50
total_collisions = 0
n_batch_seen = 0
for i, (_, _, c) in enumerate(train_loader):
    total_collisions += c
    n_batch_seen += 1
    if i + 1 >= N_SAMPLE_BATCH:
        break
print(f'Grid cell collisions di train (sample {n_batch_seen} batch): {total_collisions} '
      f'-- kalau besar, GRID_SIZE={GRID_SIZE} kurang halus utk multi-box dekat.')

train=18865  val=2358  test=2358
Grid cell collisions di train (sample 50 batch): 101 -- kalau besar, GRID_SIZE=14 kurang halus utk multi-box dekat.


## K -- Training Stage 1 (Backbone Frozen, No CBAM)

In [15]:
print('='*60)
print('STAGE 1: Backbone frozen, head only, no CBAM')
print('='*60)

model = RupiahLocalizer(pretrained=True, with_cbam=False).to(DEVICE)
model.freeze_backbone()

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

best_map_s1, best_epoch_s1, patience_ctr = 0.0, 0, 0
t0 = time.time()

for epoch in range(1, EPOCHS_STAGE1 + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    map_result = evaluate_map(model, val_loader)
    scheduler.step()

    print(f'[S1 {epoch:02d}/{EPOCHS_STAGE1}] train_loss={train_loss:.4f}  '
          f"val_mAP50={map_result['map50']:.4f}  val_mAP50-95={map_result['map50_95']:.4f}  "
          f"val_mar100={map_result['mar100']:.4f}")

    score = map_result['map50']
    if score > best_map_s1:
        best_map_s1, best_epoch_s1, patience_ctr = score, epoch, 0
        torch.save(model.state_dict(), str(PTH_STAGE1))
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE and epoch >= MIN_EPOCHS_S1:
            print(f'  Early stop di epoch {epoch} (patience={PATIENCE})')
            break

t_s1 = (time.time() - t0) / 3600
print(f'\nStage 1 selesai: best_epoch={best_epoch_s1}  best_mAP50={best_map_s1:.4f}  time={t_s1:.2f}h')


STAGE 1: Backbone frozen, head only, no CBAM


model.safetensors:   0%|          | 0.00/39.2M [00:00<?, ?B/s]

Unexpected keys (norm_head.num_batches_tracked, classifier.bias, classifier.weight, conv_head.weight, norm_head.bias, norm_head.running_mean, norm_head.running_var, norm_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


mobilenetv4_conv_medium: feat_channels=160  grid=14x14
  with_cbam=False
  total_params=7,461,909
Backbone frozen.
[S1 01/20] train_loss=15.4024  val_mAP50=0.6997  val_mAP50-95=0.3627  val_mar100=0.4371
[S1 02/20] train_loss=4.1361  val_mAP50=0.7792  val_mAP50-95=0.4352  val_mar100=0.5108
[S1 03/20] train_loss=2.9736  val_mAP50=0.8581  val_mAP50-95=0.4820  val_mar100=0.5642
[S1 04/20] train_loss=2.5399  val_mAP50=0.8779  val_mAP50-95=0.5244  val_mar100=0.6022
[S1 05/20] train_loss=2.2979  val_mAP50=0.9008  val_mAP50-95=0.5441  val_mar100=0.6223
[S1 06/20] train_loss=2.1634  val_mAP50=0.9030  val_mAP50-95=0.5616  val_mar100=0.6376
[S1 07/20] train_loss=2.0440  val_mAP50=0.9174  val_mAP50-95=0.5643  val_mar100=0.6364
[S1 08/20] train_loss=1.9584  val_mAP50=0.9221  val_mAP50-95=0.5707  val_mar100=0.6439
[S1 09/20] train_loss=1.8992  val_mAP50=0.9199  val_mAP50-95=0.5830  val_mar100=0.6535
[S1 10/20] train_loss=1.8411  val_mAP50=0.9290  val_mAP50-95=0.5961  val_mar100=0.6607
[S1 11/20] tra

## L -- Training Stage 2 (Backbone Unfrozen + CBAM)

In [16]:
print('='*60)
print('STAGE 2: Backbone unfrozen, CBAM injected')
print('='*60)

model = RupiahLocalizer(pretrained=False, with_cbam=USE_CBAM).to(DEVICE)
state_s1 = torch.load(str(PTH_STAGE1), map_location=DEVICE)
model.load_state_dict(state_s1, strict=False)   # strict=False krn CBAM baru ditambah
model.unfreeze_backbone()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_S2, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

best_map_s2, best_epoch_s2, patience_ctr = 0.0, 0, 0
t0 = time.time()

for epoch in range(1, EPOCHS_STAGE2 + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    map_result = evaluate_map(model, val_loader)
    scheduler.step()

    print(f'[S2 {epoch:02d}/{EPOCHS_STAGE2}] train_loss={train_loss:.4f}  '
          f"val_mAP50={map_result['map50']:.4f}  val_mAP50-95={map_result['map50_95']:.4f}  "
          f"val_mar100={map_result['mar100']:.4f}")

    score = map_result['map50']
    if score > best_map_s2:
        best_map_s2, best_epoch_s2, patience_ctr = score, epoch, 0
        torch.save(model.state_dict(), str(PTH_STAGE2))
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE and epoch >= MIN_EPOCHS_S2:
            print(f'  Early stop di epoch {epoch} (patience={PATIENCE})')
            break

t_s2 = (time.time() - t0) / 3600
print(f'\nStage 2 selesai: best_epoch={best_epoch_s2}  best_mAP50={best_map_s2:.4f}  time={t_s2:.2f}h')


STAGE 2: Backbone unfrozen, CBAM injected
mobilenetv4_conv_medium: feat_channels=160  grid=14x14
  with_cbam=True
  total_params=7,465,207
Backbone unfrozen: 7,203,152 params
[S2 01/40] train_loss=1.8695  val_mAP50=0.9535  val_mAP50-95=0.6361  val_mar100=0.7000
[S2 02/40] train_loss=1.4893  val_mAP50=0.9714  val_mAP50-95=0.6821  val_mar100=0.7381
[S2 03/40] train_loss=1.3736  val_mAP50=0.9749  val_mAP50-95=0.7044  val_mar100=0.7586
[S2 04/40] train_loss=1.3103  val_mAP50=0.9717  val_mAP50-95=0.7145  val_mar100=0.7678
[S2 05/40] train_loss=1.2487  val_mAP50=0.9740  val_mAP50-95=0.7340  val_mar100=0.7866
[S2 06/40] train_loss=1.1892  val_mAP50=0.9759  val_mAP50-95=0.7490  val_mar100=0.8027
[S2 07/40] train_loss=1.1477  val_mAP50=0.9743  val_mAP50-95=0.7464  val_mar100=0.7988
[S2 08/40] train_loss=1.1130  val_mAP50=0.9772  val_mAP50-95=0.7648  val_mar100=0.8136
[S2 09/40] train_loss=1.0784  val_mAP50=0.9770  val_mAP50-95=0.7720  val_mar100=0.8198
[S2 10/40] train_loss=1.0466  val_mAP50=0.

## M -- Final Evaluate (Test Set): mAP + teacher-forced + visualisasi

In [17]:
model = RupiahLocalizer(pretrained=False, with_cbam=USE_CBAM).to(DEVICE)
model.load_state_dict(torch.load(str(PTH_STAGE2), map_location=DEVICE))
model.eval()

test_map = evaluate_map(model, test_loader)
_, test_iou_tf, test_recall_tf = evaluate_teacher_forced(model, test_loader)

print('TEST -- metrik UTAMA (mAP, false positive dihitung):')
print(f"  mAP50    = {test_map['map50']:.4f}")
print(f"  mAP50-95 = {test_map['map50_95']:.4f}")
print(f"  mAR100   = {test_map['mar100']:.4f}")
print('\nTEST -- metrik historis (teacher-forced, JANGAN dibandingkan ke mAP YOLO):')
print(f'  mean_IoU (GT-cell only) = {test_iou_tf:.4f}')
print(f'  recall (GT-cell only)   = {test_recall_tf:.4f}')

with open(CSV_PATH, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['metric', 'value'])
    w.writerow(['test_map50', test_map['map50']])
    w.writerow(['test_map50_95', test_map['map50_95']])
    w.writerow(['test_mar100', test_map['mar100']])
    w.writerow(['test_iou_teacher_forced', test_iou_tf])
    w.writerow(['test_recall_teacher_forced', test_recall_tf])

# -- Visualisasi sample prediksi --------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
SAMPLE_SEED = 123
random.seed(SAMPLE_SEED)
sample_idx = random.sample(range(len(test_ds)), min(8, len(test_ds)))

for ax, idx in zip(axes.flat, sample_idx):
    img_t, target, _ = test_ds[idx]
    with torch.no_grad():
        pred = model(img_t.unsqueeze(0).to(DEVICE))[0].cpu()
    raw_boxes = decode_boxes(pred, conf_thresh=CONF_THRESHOLD)
    final_boxes = nms_plain(raw_boxes)

    img_np = img_t.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    ax.imshow(img_np)
    for conf, cx, cy, w, h in final_boxes:
        x = (cx - w/2) * IMG_SIZE
        y = (cy - h/2) * IMG_SIZE
        rect = mpatches.Rectangle((x, y), w*IMG_SIZE, h*IMG_SIZE,
                                    linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y-3, f'{conf:.2f}', color='lime', fontsize=9)
    ax.set_title(f'pred={len(final_boxes)} box', fontsize=10)
    ax.axis('off')

plt.tight_layout()
VIZ_PATH = OUT / 'sample_predictions.png'
plt.savefig(VIZ_PATH, dpi=100)
plt.show()
print(f'Saved: {VIZ_PATH}')


mobilenetv4_conv_medium: feat_channels=160  grid=14x14
  with_cbam=True
  total_params=7,465,207
TEST -- metrik UTAMA (mAP, false positive dihitung):
  mAP50    = 0.9891
  mAP50-95 = 0.8279
  mAR100   = 0.8717

TEST -- metrik historis (teacher-forced, JANGAN dibandingkan ke mAP YOLO):
  mean_IoU (GT-cell only) = 0.9190
  recall (GT-cell only)   = 0.8206
Saved: /kaggle/working/outputs/sample_predictions.png


## M2 -- Eval end-to-end (localizer -> crop -> classifier v15) (FASE 0 item 3)

In [18]:
# FASE 0 item 3: eval end-to-end (localizer -> crop persis kode app ->
# classifier v15 -> akurasi akhir). Ini yang paling hilang dari notebook lama --
# ketidakcocokan cara crop antara training dan app baru kelihatan sebagai
# akurasi jelek di HP, dan tidak muncul di evaluasi masing-masing model sendiri.
#
# BUTUH classifier_v15.pth / TFLite sebagai Kaggle dataset input -- tidak ada
# di environment ini, jadi diisi path-nya manual sebelum run. Kalau path belum
# ada, sel ini sengaja print instruksi dan berhenti, bukan pura-pura jalan.

CLASSIFIER_CKPT_PATH = Path('/kaggle/input/PASANG-DATASET-CLASSIFIER-V15-DI-SINI/best_stage2_v15.pth')
# CROP_MARGIN harus SAMA PERSIS dengan classifier v15 Cell B/E -- jangan diketik
# ulang manual, import dari situ kalau notebook classifier di-attach sebagai
# Kaggle dataset/utility script. Nilai di bawah cuma fallback kalau tidak diimport.
CROP_MARGIN_CLS = 0.12

if not CLASSIFIER_CKPT_PATH.exists():
    print('SKIP eval end-to-end: classifier checkpoint belum di-attach.')
    print('Cara pasang: Kaggle -> Add Input -> upload best_stage2_v15.pth (dan')
    print('CLASS_NAMES classifier v15) sebagai dataset, lalu ganti CLASSIFIER_CKPT_PATH')
    print('di atas. Tanpa ini, akurasi end-to-end TIDAK bisa diverifikasi -- jangan')
    print('klaim pipeline gabungan bekerja hanya berdasarkan angka localizer sendiri.')
else:
    # Import class classifier -- asumsi RupiahClassifier persis definisi v15
    # (Cell I di classifier notebook). Salin definisi itu ke sel sebelum ini
    # kalau dijalankan sebagai notebook berdiri sendiri.
    CLASSIFIER_CLASS_NAMES = [
        'Rp100_koin', 'Rp200_koin', 'Rp500_koin', 'Rp1000_koin',
        'Rp1000_kertas', 'Rp2000_kertas', 'Rp5000_kertas', 'Rp10000_kertas',
        'Rp20000_kertas', 'Rp50000_kertas', 'Rp100000_kertas',
    ]
    clf = RupiahClassifier(num_classes=len(CLASSIFIER_CLASS_NAMES), with_cbam=True).to(DEVICE)
    clf.load_state_dict(torch.load(str(CLASSIFIER_CKPT_PATH), map_location=DEVICE))
    clf.eval()

    def crop_and_letterbox_eval(image, box, margin=CROP_MARGIN_CLS, size=256):
        H, W = image.shape[:2]
        cx, cy, bw, bh = box
        x1 = max(0, int((cx - bw/2 - margin) * W))
        y1 = max(0, int((cy - bh/2 - margin) * H))
        x2 = min(W, int((cx + bw/2 + margin) * W))
        y2 = min(H, int((cy + bh/2 + margin) * H))
        crop = image[y1:y2, x1:x2]
        h, w = crop.shape[:2]
        scale = size / max(h, w, 1)
        crop = cv2.resize(crop, (max(1,int(w*scale)), max(1,int(h*scale))))
        pad_h, pad_w = size - crop.shape[0], size - crop.shape[1]
        crop = cv2.copyMakeBorder(crop, pad_h//2, pad_h - pad_h//2,
                                   pad_w//2, pad_w - pad_w//2,
                                   cv2.BORDER_CONSTANT, value=(114,114,114))
        return crop

    cls_eval_tf = A.Compose([
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

    n_correct_e2e, n_total_e2e, n_localizer_missed = 0, 0, 0
    for idx in range(len(test_ds)):
        img_path = test_ds.files[idx]
        lbl_path = test_ds.lbl_dir / (img_path.stem + '.txt')
        gt_lines = open(lbl_path).read().splitlines()
        if not gt_lines:
            continue
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        img_t, _, _ = test_ds[idx]
        with torch.no_grad():
            pred = model(img_t.unsqueeze(0).to(DEVICE))[0].cpu()
        boxes = nms_plain(decode_boxes(pred, conf_thresh=CONF_THRESHOLD))
        n_total_e2e += 1
        if not boxes:
            n_localizer_missed += 1
            continue
        conf, cx, cy, bw, bh = max(boxes, key=lambda b: b[0])
        crop = crop_and_letterbox_eval(image, (cx, cy, bw, bh))
        crop_t = cls_eval_tf(image=crop)['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = clf(crop_t)
        pred_cls = CLASSIFIER_CLASS_NAMES[logits.argmax(1).item()]
        # TODO: bandingkan pred_cls ke label kelas asli gambar ini -- test_ds
        # localizer TIDAK punya info nominal (single class), jadi ground-truth
        # nominal harus diambil dari nama folder/source asal gambar sebelum
        # collapse ke "uang". Simpan orig_name->nominal saat Cell E kalau mau
        # sel ini menghitung akurasi asli, bukan cuma "localizer nemu box".

    print(f'Localizer gagal deteksi sama sekali: {n_localizer_missed}/{n_total_e2e} '
          f'({n_localizer_missed/max(n_total_e2e,1):.1%})')
    print('Akurasi klasifikasi end-to-end: lihat TODO di atas -- butuh mapping '
          'nominal asli per gambar test, belum tersedia di dataset single-class ini.')


SKIP eval end-to-end: classifier checkpoint belum di-attach.
Cara pasang: Kaggle -> Add Input -> upload best_stage2_v15.pth (dan
CLASS_NAMES classifier v15) sebagai dataset, lalu ganti CLASSIFIER_CKPT_PATH
di atas. Tanpa ini, akurasi end-to-end TIDAK bisa diverifikasi -- jangan
klaim pipeline gabungan bekerja hanya berdasarkan angka localizer sendiri.


## N -- Export ONNX + TFLite FP32 (normalisasi dipanggang ke graph, FASE 4)

In [21]:
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'onnxscript', 'onnx', 'onnxsim', '-q'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'onnx2tf', 'sng4onnx', 'onnx_graphsurgeon',
                'tensorflow', '-q', '--no-warn-conflicts'], check=False)

import onnx

class LocalizerExportWrapper(nn.Module):
    '''
    FASE 4 item 2: normalisasi ImageNet DIPANGGANG ke dalam graph -- v3 minta
    sisi RN kirim tensor yang sudah dinormalisasi manual, itu satu sumber bug
    kalau ada mismatch (layout NCHW/NHWC juga rawan, lihat catatan validasi di
    bawah). Sekarang graph terima RGB float 0..1 langsung dari decode gambar,
    satu sumber kebenaran.

    Objectness di-sigmoid di sini (raw dari model tetap logit buat loss numerik
    stabil saat training). Box SUDAH absolut 0..1 dari model.forward() (Cell H) --
    tidak ada decode grid manual lagi di sisi app, cuma NMS.
    '''
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.mean) / self.std
        raw = self.model(x)
        obj = torch.sigmoid(raw[..., 0:1])
        box = raw[..., 1:5]
        return torch.cat([obj, box], dim=-1)


print('Preparing export model...')
export_inner = RupiahLocalizer(pretrained=False, with_cbam=USE_CBAM)
export_inner.load_state_dict(torch.load(str(PTH_STAGE2), map_location='cpu'))
export_inner.eval().cpu()

wrapper = LocalizerExportWrapper(export_inner).eval().cpu()
dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)   # 0..1, RGB -- match app input sekarang
with torch.no_grad():
    out = wrapper(dummy)
print(f'Wrapper OK: {out.shape}  (obj range {out[...,0].min():.3f}..{out[...,0].max():.3f}, '
      f'box range {out[...,1:].min():.3f}..{out[...,1:].max():.3f})')

print(f'\n[1/2] ONNX -> {ONNX_PATH}')
torch.onnx.export(
    wrapper, dummy, str(ONNX_PATH),
    opset_version=17,
    dynamo=False,
    input_names=['input'], output_names=['detections'],
    export_params=True, do_constant_folding=True, dynamic_axes=None)
onnx_mb = ONNX_PATH.stat().st_size / 1e6
print(f'  ONNX: {onnx_mb:.1f} MB')

print(f'\n[2/2] TFLite FP32 -> {TFLITE_PATH}')
tflite_ok = False
onnx_sim = OUT / (ONNX_PATH.stem + '_sim.onnx')

try:
    import importlib, onnx2tf
    importlib.reload(onnx2tf)
    from onnxsim import simplify

    model_onnx = onnx.load(str(ONNX_PATH))
    model_sim, ok = simplify(model_onnx)
    onnx.save(model_sim if ok else model_onnx, str(onnx_sim))
    print(f'  ONNX sim: {onnx_sim.stat().st_size/1e6:.1f} MB  ok={ok}')

    tflite_dir = OUT / 'tflite_fp32_out'
    tflite_dir.mkdir(exist_ok=True)
    onnx2tf.convert(
        input_onnx_file_path=str(onnx_sim),
        output_folder_path=str(tflite_dir),
        non_verbose=True,
        output_integer_quantized_tflite=False,
        keep_shape_absolutely_input_names=['input'])

    fp32_files = [f for f in tflite_dir.glob('*.tflite') if 'float16' not in f.name]
    chosen = fp32_files[0] if fp32_files else (list(tflite_dir.glob('*.tflite')) or [None])[0]
    if chosen:
        shutil.copy2(str(chosen), str(TFLITE_PATH))
        tflite_mb = TFLITE_PATH.stat().st_size / 1e6
        tflite_ok = True
        print(f'  TFLite FP32: {tflite_mb:.1f} MB  (from {chosen.name})')
    else:
        print('  onnx2tf ran tapi tidak ada .tflite output')
except Exception as e:
    print(f'  onnx2tf failed: {e}')
    import traceback; traceback.print_exc()

print(f'\nExport summary ({MODEL_NAME}):')
print(f'  ONNX        : {onnx_mb:.1f} MB')
if tflite_ok:
    print(f'  TFLite FP32 : {tflite_mb:.1f} MB')
else:
    print('  TFLite FP32 : GAGAL -- cek error di atas.')


Preparing export model...
mobilenetv4_conv_medium: feat_channels=160  grid=14x14
  with_cbam=True
  total_params=7,465,207
Wrapper OK: torch.Size([1, 14, 14, 5])  (obj range 0.000..0.260, box range 0.039..0.958)

[1/2] ONNX -> /kaggle/working/outputs/localizer_v4.onnx
  ONNX: 7.5 MB

[2/2] TFLite FP32 -> /kaggle/working/outputs/localizer_v4.tflite
  ONNX sim: 7.5 MB  ok=True


flatbuffer_direct lowering:   0%|          | 0/115 [00:00<?, ?it/s]

flatbuffer_direct post-lowering:   0%|          | 0/6 [00:00<?, ?it/s]

flatbuffer_direct export:   0%|          | 0/3 [00:00<?, ?it/s]

flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.030s serialize=0.026s (sanitize=0.000s build=0.003s pack=0.018s output=0.005s) write=0.004s size=7.20MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.024s serialize=0.022s (sanitize=0.000s build=0.002s pack=0.017s output=0.002s) write=0.002s size=3.62MB
  TFLite FP32: 7.6 MB  (from localizer_v4_sim_float32.tflite)

Export summary (Localizer_MobileNetV4_SingleClass_v4):
  ONNX        : 7.5 MB
  TFLite FP32 : 7.6 MB


## O -- Kuantisasi INT8 (FASE 4 item 3)

In [29]:
# FASE 4 item 3: kuantisasi INT8 penuh. Dugaan di plan: ini sumber speedup
# lebih besar daripada mengecilkan backbone (conv_medium -> conv_small).
# WAJIB dibandingkan mAP FP32 vs INT8 di bawah sebelum dipakai -- jangan
# diasumsikan "pasti aman" cuma karena berhasil convert.
#
# v3: onnx2tf di environment ini TIDAK menghasilkan folder saved_model sama
# sekali (terbukti dari isi tflite_fp32_out: cuma *_float32.tflite dan
# *_float16.tflite langsung). Jadi TFLiteConverter.from_saved_model() salah
# pendekatan dari awal untuk versi onnx2tf ini -- bukan soal path yang keliru.
# Fix: minta onnx2tf SENDIRI yang menghasilkan versi INT8, lewat parameter
# output_integer_quantized_tflite + custom_input_op_name_np_data_path, persis
# jalur yang sama yang berhasil menghasilkan FP32 di Cell N.
#
# Tetap di subprocess terpisah (protobuf pin) sesuai pola Cell N/P.

import subprocess, sys, textwrap
from pathlib import Path

onnx_sim_path = OUT / (ONNX_PATH.stem + '_sim.onnx')
int8_dir = OUT / 'tflite_int8_out'
rep_npy_path = OUT / 'representative_data.npy'

int8_script = textwrap.dedent(f'''
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "protobuf==6.31.1", "-q"], check=False)
    import numpy as np, cv2
    from pathlib import Path
    import onnx2tf

    # -- siapkan representative dataset sebagai satu file .npy, NHWC 0..1 --
    # (v4 fix: onnx2tf minta layout NHWC utk kalibrasi -- mean/std shape (3,)
    # cuma bisa broadcast kalau channel di sumbu terakhir. Error sebelumnya
    # "could not broadcast (200,3,224,224) (3,)" itu bukti layoutnya salah.)
    rep_files = sorted(Path(r"{DATA_DIR / 'train' / 'images'}").glob("*"))[:200]
    imgs = []
    for img_path in rep_files:
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, ({IMG_SIZE}, {IMG_SIZE})).astype(np.float32) / 255.0
        imgs.append(img)   # HWC -- JANGAN transpose ke CHW
    rep_array = np.stack(imgs)   # [N, {IMG_SIZE}, {IMG_SIZE}, 3]
    np.save(r"{rep_npy_path}", rep_array)
    print("Representative data:", rep_array.shape)

    onnx2tf.convert(
        input_onnx_file_path=r"{onnx_sim_path}",
        output_folder_path=r"{int8_dir}",
        non_verbose=True,
        output_integer_quantized_tflite=True,
        # mean=0, std=1 -- normalisasi ImageNet SUDAH dipanggang di graph
        # (LocalizerExportWrapper), jadi data kalibrasi di sini raw 0..1 saja.
        custom_input_op_name_np_data_path=[["input", r"{rep_npy_path}", [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]]],
        keep_shape_absolutely_input_names=["input"],
    )

    int8_files = [p for p in Path(r"{int8_dir}").rglob("*.tflite")
                  if "int8" in p.name.lower() or "integer" in p.name.lower() or "quant" in p.name.lower()]
    if not int8_files:
        print("Tidak ketemu file INT8 spesifik, isi folder:")
        for p in sorted(Path(r"{int8_dir}").rglob("*.tflite")):
            print(" ", p.name)
        raise FileNotFoundError("onnx2tf tidak menghasilkan file bertanda int8/quant -- cek listing di atas.")

    chosen = int8_files[0]
    Path(r"{TFLITE_INT8_PATH}").write_bytes(chosen.read_bytes())
    print("INT8 convert OK, sumber:", chosen.name, "-", chosen.stat().st_size / 1e6, "MB")
''')

script_path = Path('/kaggle/working/_int8_quant.py')
script_path.write_text(int8_script)

result = subprocess.run([sys.executable, str(script_path)], capture_output=True, text=True)
print(result.stdout)

int8_ok = TFLITE_INT8_PATH.exists()
if result.returncode != 0 or not int8_ok:
    print('INT8 quantization GAGAL, stderr:')
    print(result.stderr[-2500:])
    print('Fallback: pakai TFLite FP32 saja. Ini bukan blocker Fase 4, cuma '
          'kehilangan potensi speedup -- catat di laporan, jangan diam-diam skip.')
else:
    fp32_mb = TFLITE_PATH.stat().st_size / 1e6
    int8_mb = TFLITE_INT8_PATH.stat().st_size / 1e6
    print(f'TFLite INT8: {int8_mb:.1f} MB  (FP32 was {fp32_mb:.1f} MB)')
    print('\nWAJIB: bandingkan mAP FP32 vs INT8 sebelum dipakai di app.')
    print('Ganti TFLITE_PATH -> TFLITE_INT8_PATH lalu jalankan ulang loop mAP')
    print('pakai tf.lite.Interpreter (bukan model PyTorch) di test set.')
    print('Penurunan wajar: <1 poin mAP50. Kalau lebih besar, representative')
    print('dataset kemungkinan kurang mewakili -- perbesar dari 200 sample.')

Representative data: (200, 224, 224, 3)
flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.037s serialize=0.030s (sanitize=0.000s build=0.003s pack=0.021s output=0.005s) write=0.007s size=7.21MB
flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.035s serialize=0.028s (sanitize=0.000s build=0.003s pack=0.019s output=0.006s) write=0.006s size=7.21MB

INT8 quantization GAGAL, stderr:
direct export [2/7] write float32 tflite:  29%|██▊       | 2/7 [00:00<00:00,  6.10it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnx2tf/tflite_builder/quantization.py", line 418, in collect_calibration_ranges_from_tflite
    interpreter.allocate_tensors()
  File "/usr/local/lib/python3.12/dist-packages/ai_edge_litert/interpreter.py", line 554, in allocate_tensors
    return self._interpreter.AllocateTensors()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: Given shapes, [1,224,224,3] and [1,3,1,1], are not br

## P -- Validate TFLite + Package Outputs

In [30]:
import subprocess, sys, textwrap
from pathlib import Path

validate_script = textwrap.dedent(f'''
    import subprocess, sys, os
    subprocess.run([sys.executable, "-m", "pip", "install", "protobuf==6.31.1", "-q"], check=False)
    import numpy as np
    import tensorflow as tf

    paths = {{"FP32": "{TFLITE_PATH}", "INT8": "{TFLITE_INT8_PATH}"}}
    for label, path in paths.items():
        if not os.path.exists(path):
            print(label, ": file tidak ada, skip")
            continue
        interp = tf.lite.Interpreter(model_path=path)
        interp.allocate_tensors()
        inp_d = interp.get_input_details()
        out_d = interp.get_output_details()
        in_shape, in_dtype = inp_d[0]["shape"], inp_d[0]["dtype"]
        out_shape, out_dtype = out_d[0]["shape"], out_d[0]["dtype"]
        print(label, "-- input :", in_shape, in_dtype)
        print(label, "-- output:", out_shape, out_dtype, "(expect [1, {GRID_SIZE}, {GRID_SIZE}, 5])")

        dummy_np = np.random.rand(1, 3, {IMG_SIZE}, {IMG_SIZE}).astype(np.float32)
        if in_dtype == np.uint8:
            dummy_np = (dummy_np * 255).astype(np.uint8)
        interp.set_tensor(inp_d[0]["index"], dummy_np)
        interp.invoke()
        det = interp.get_tensor(out_d[0]["index"])
        print(label, "OK: output shape", det.shape)

    print()
    print("PENTING: NMS dan filter confidence dikerjakan di RN side, bukan di model.")
    print("Input sekarang RGB float 0..1 (FP32) atau uint8 (INT8) -- TIDAK perlu")
    print("normalisasi ImageNet manual di RN lagi, sudah dipanggang ke graph.")
''')

script_path = Path('/kaggle/working/_validate_tflite.py')
script_path.write_text(validate_script)

result = subprocess.run([sys.executable, str(script_path)], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('VALIDASI GAGAL, stderr:')
    print(result.stderr[-2000:])

output_files = [
    ('localizer_stage1.pth', PTH_STAGE1),
    ('localizer_stage2.pth', PTH_STAGE2),
    ('localizer.onnx', ONNX_PATH),
    ('localizer_fp32.tflite', TFLITE_PATH),
    ('localizer_int8.tflite', TFLITE_INT8_PATH),
    ('results.csv', CSV_PATH),
    ('sample_predictions.png', OUT / 'sample_predictions.png'),
]
zip_path = OUT / f'localizer_mobilenetv4_singleclass_{VERSION}.zip'
with zipfile.ZipFile(str(zip_path), 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname, fpath in output_files:
        if Path(fpath).exists():
            zf.write(str(fpath), fname)
            print(f'  + {fname:<30} ({Path(fpath).stat().st_size/1e6:.1f} MB)')

print(f'\nZIP: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)')
print('\n' + '='*60)
print(f'FINAL SUMMARY -- {MODEL_NAME}')
print('='*60)
print(f'  S1 best epoch : {best_epoch_s1}  mAP50={best_map_s1:.4f}')
print(f'  S2 best epoch : {best_epoch_s2}  mAP50={best_map_s2:.4f}')
print(f"  Test mAP50    : {test_map['map50']:.4f}")
print(f"  Test mAP50-95 : {test_map['map50_95']:.4f}")
print('='*60)

from IPython.display import FileLink, display
display(FileLink(str(zip_path.relative_to('/kaggle/working'))))


FP32 -- input : [  1   3 224 224] <class 'numpy.float32'>
FP32 -- output: [ 1 14 14  5] <class 'numpy.float32'> (expect [1, 14, 14, 5])
FP32 OK: output shape (1, 14, 14, 5)
INT8 : file tidak ada, skip

PENTING: NMS dan filter confidence dikerjakan di RN side, bukan di model.
Input sekarang RGB float 0..1 (FP32) atau uint8 (INT8) -- TIDAK perlu
normalisasi ImageNet manual di RN lagi, sudah dipanggang ke graph.

  + localizer_stage1.pth           (30.3 MB)
  + localizer_stage2.pth           (30.3 MB)
  + localizer.onnx                 (7.5 MB)
  + localizer_fp32.tflite          (7.6 MB)
  + results.csv                    (0.0 MB)
  + sample_predictions.png         (1.5 MB)

ZIP: /kaggle/working/outputs/localizer_mobilenetv4_singleclass_v4.zip (72.1 MB)

FINAL SUMMARY -- Localizer_MobileNetV4_SingleClass_v4
  S1 best epoch : 16  mAP50=0.9457
  S2 best epoch : 30  mAP50=0.9893
  Test mAP50    : 0.9891
  Test mAP50-95 : 0.8279


/kaggle/working/outputs/localizer_mobilenetv4_singleclass_v4.zip